# 07 - Reversibility Analysis

This notebook evaluates electrochemical and chemical reversibility for a real scan-rate series. It uses the bundled Fe/Fc data to show two distinct outcomes: a reversible Fc reference wave and a quasi-reversible Fe wave. The analysis reports the evidence and thresholds behind each cautious series-level conclusion.

## Import eCAT And Set Paths

The notebook uses only core eCAT and bundled real CV files. Potentials are referenced to Fc/Fc+ during import.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
ROOT = ROOT.parent
DATA_DIR = ROOT / 'examples' / 'data' / 'fe_phoh_cv'

import ecat as e

print("eCAT version:", getattr(e, "__version__", "unknown"))


eCAT version: 0.1.0b6


## Load A Real Scan-Rate Series

The five Ar CVs share composition and scan window but span 0.025 to 1 V/s. The electrode diameter supplies area metadata, and filename parsing supplies the Fc and Fe concentrations used by the optional Sevcik diffusion estimate.

In [2]:
all_cvs = e.get_data({
    'folder path': str(DATA_DIR),
    'reference mode': 'keyword',
    'reference keyword': 'Fc',
    'reference guess': 0.4,
    'reference label': 'Fc/Fc+',
    'electrode diameter': 0.3,
    'print': False,
})
three_segment_cvs = e.filter(all_cvs, {'segments': 3}, {'print': False})
ar_fe_cvs = e.filter(
    three_segment_cvs,
    {'gas': 'Ar', 'compounds': ['Fc', 'Fe-tpyPY2Me']},
    {'logic': 'AND', 'print': False},
)
scan_series = e.filter(ar_fe_cvs, {'scan window': [-1.7, 1]}, {'print': False})
scan_series = e.sort(scan_series, 'scan rate', {'print': False})

assert len(scan_series) == 5
e.show(scan_series)


[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Scan Rate
[0],25 mV/s
[1],50 mV/s
[2],100 mV/s
[3],500 mV/s
[4],1 V/s


## Inspect The Series

A scan-rate colorbar makes ordering explicit. The same traces contain the Fc wave near 0 V and the Fe wave near -1.5 V after referencing.

In [3]:
e.multiplot(scan_series, {
    'print': False,
    'title': 'Real Fe/Fc Scan-Rate Series',
    'gradient by': 'scan rate',
    'legend mode': 'colorbar',
    'colorbar tick labels': 'all',
})


<Axes: title={'center': 'Real Fe/Fc Scan-Rate Series'}, xlabel='Potential (V vs $\\mathrm{Fc/Fc^{+}}$)', ylabel='Current (μA)'>

## Individual-CV Evidence

`wave_info()` provides the single-CV evidence consumed across the series: $E_{1/2}$, $\Delta E_p$, $i_{p,\mathrm{c}}$, $i_{p,\mathrm{a}}$, $|i_{p,\mathrm{a}}/i_{p,\mathrm{c}}|$, and tangent-corrected half-peak widths when those widths can be resolved. No reversibility label is assigned from one CV.

In [4]:
representative = scan_series[2]
fc_wave = representative.wave_info({
    'segments': [2, 3],
    'guess potential': 0.0,
    'plot': False,
    'print': True,
})
fe_wave = representative.wave_info({
    'segments': [1, 2],
    'guess potential': -1.5,
    'plot': False,
    'print': True,
})


Metric,Segment,Value
E1/2,,-2.776e-14 mV
ΔEp,,57.00 mV
"Ep,a",2,28.50 mV
"ip,a",2,119.7 μA
"Ep/2,a",2,-28.50 mV
"Δ(Ep,a - Ep/2,a)",2,57.00 mV
"W1/2,a",2,226.4 mV
"Ep,c",3,-28.50 mV
"ip,c",3,-117.6 μA
"Ep/2,c",3,28.50 mV


Metric,Segment,Value
E1/2,,-1.423 V
ΔEp,,0.1160 V
"Ep,c",1,-1.481 V
"ip,c",1,-40.07 μA
"Ep/2,c",1,-1.377 V
"Δ(Ep,c - Ep/2,c)",1,-0.1040 V
"W1/2,c",1,0.3500 V
"Ep,a",2,-1.365 V
"ip,a",2,38.10 μA
"Ep/2,a",2,-1.467 V


## Reversible Fc Reference Wave

Omitting `D` while providing `species='Fc'` asks eCAT to estimate branch-specific $D_{\mathrm{app}}$ values from the Sevcik slopes. The Matsuda-Ayabe classification then uses $\Lambda$ independently of whether a point lies in Nicholson's recommended rate-estimation range. A fully reversible-looking series yields a lower bound on $k^0$.

In [5]:
fc_reversibility = e.reversibility_analysis(
    scan_series,
    {
        'phase': 'bulk',
        'segments': [2, 3],
        'guess potential': 0.0,
        'species': 'Fc',
        'num electrons': 1,
        'plot': True,
        'plot all': True,
        'print': True,
    },
)


Parameter,Symbol,Value
Phase,,bulk
Electron Count,n,1.000000
Temperature,T,298 K
Diffusion Coefficient,D,4.224e-05 cm2/s
Electrode Area,S,0.07069 cm2
Scan Rates,ν,0.025 to 1 V/s
Segments,,"[2, 3]"
Agreement Tolerance,,0.25
Current Ratio Tolerance,,0.1


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Assessment,Conclusion,Evidence
Electron Transfer,reversible,"Matsuda-Ayabe Λ=23.99 to 23.99, with additional values above the reversible limit across 0.025 to 1 V/s: 5 reversible scan-rate means"
Chemical Reversibility,chemically reversible over observed timescale,"all 5 tangent-corrected |ip,a/ip,c| ratios are within tolerance of unity (range 0.9696 to 1.059; maximum deviation from unity is 0.05899; current-ratio tolerance is 0.1)"
Electron-Transfer Rate,reversible lower bound,>= 0.6084 cm/s


## Quasi-Reversible Fe Wave

The Fe wave uses the same real CVs but different physical segments and peak guess. Its concentration metadata supports the same automatic branch-Sevcik estimate. Here the Nicholson-eligible points provide an estimated heterogeneous $k^0$.

In [6]:
fe_reversibility = e.reversibility_analysis(
    scan_series,
    {
        'phase': 'bulk',
        'segments': [1, 2],
        'guess potential': -1.5,
        'species': 'Fe-tpyPY2Me',
        'num electrons': 1,
        'plot': True,
        'plot all': True,
        'print': True,
    },
)


Parameter,Symbol,Value
Phase,,bulk
Electron Count,n,1.000000
Temperature,T,298 K
Diffusion Coefficient,D,3.304e-05 cm2/s
Electrode Area,S,0.07069 cm2
Scan Rates,ν,0.025 to 1 V/s
Segments,,"[1, 2]"
Agreement Tolerance,,0.25
Current Ratio Tolerance,,0.1


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Assessment,Conclusion,Evidence
Electron Transfer,quasi-reversible,Matsuda-Ayabe Λ=0.5771 to 0.7024 across 0.025 to 1 V/s: 5 quasi-reversible scan-rate means
Chemical Reversibility,chemically reversible over observed timescale,"all 5 tangent-corrected |ip,a/ip,c| ratios are within tolerance of unity (range 0.9267 to 0.9647; maximum deviation from unity is 0.07333; current-ratio tolerance is 0.1)"
Electron-Transfer Rate,Nicholson,0.01063 cm/s from 5 Nicholson-eligible scan rates


## Compare The Series-Level Conclusions

The labels stay cautious and separate electron-transfer behavior from evidence for coupled chemistry. Rate values are estimates or lower bounds, never presented as exact.

In [7]:
comparison = pd.DataFrame([
    {
        'Wave': 'Fc',
        'Electron Transfer': fc_reversibility.summary['electrochemical conclusion'],
        'Chemical Behavior': fc_reversibility.summary['chemical conclusion'],
        'Dapp / cm2 s-1': fc_reversibility.summary['D / cm^2 s^-1'],
        'k0 / cm s-1': fc_reversibility.summary['k0 / cm s^-1'],
        'k0 lower bound / cm s-1': fc_reversibility.summary['k0 lower bound / cm s^-1'],
    },
    {
        'Wave': 'Fe',
        'Electron Transfer': fe_reversibility.summary['electrochemical conclusion'],
        'Chemical Behavior': fe_reversibility.summary['chemical conclusion'],
        'Dapp / cm2 s-1': fe_reversibility.summary['D / cm^2 s^-1'],
        'k0 / cm s-1': fe_reversibility.summary['k0 / cm s^-1'],
        'k0 lower bound / cm s-1': fe_reversibility.summary['k0 lower bound / cm s^-1'],
    },
])
display(comparison)


,Wave,Electron Transfer,Chemical Behavior,Dapp / cm2 s-1,k0 / cm s-1,k0 lower bound / cm s-1
0,Fc,reversible,chemically reversible over observed timescale,0.000042,NaN,0.608385
1,Fe,quasi-reversible,chemically reversible over observed timescale,0.000033,0.010627,NaN


## Decision Order

eCAT applies a fixed, documented order: verify at least three distinct scan rates; resolve a consistent cathodic/anodic branch pair; estimate or accept $D$; classify electron-transfer behavior with Matsuda-Ayabe $\Lambda$; evaluate tangent-corrected $|i_{p,\mathrm{a}}/i_{p,\mathrm{c}}|$ for chemical reversibility; then report a Nicholson estimate, trumpet estimate, reversible lower bound, or unresolved rate. The result stores exact criteria in `.summary` and detailed evidence in `.diagnostics`.

## Inspect Available Options

Use the function-specific option table when adapting the workflow to another couple.

In [8]:
e.describe_options('reversibility_analysis')

,Category,Option,Default,Type,Choices,Description
0,Data/input,species,None,str or None,,"Chemical species used to resolve concentration or metadata. For normalize, exact-matches cv.compounds and pulls the paired cv.concentrations value when C/C unit are omitted."
1,Data/input,temperature,None,float or None,,"Temperature in K. If omitted, uses CV metadata and then the established 298.15 K fallback."
2,Selection/filtering,exact potential,None,float or list[float] or None,,"Exact potential for current extraction; when provided it bypasses peak-location auto-selection. In complex CV analyses, the plural alias 'exact potentials' accepts per-CV values."
3,Selection/filtering,guess potential,None,float or list[float] or list[list[float]] or None,,"Initial potential guess for automatic peak or wave selection. In complex CV analyses, the plural alias 'guess potentials' accepts per-CV values; scalar guesses keep running-guess behavior where supported."
4,Selection/filtering,plot segment,None,int or None,,Segment to emphasize or plot.
5,Selection/filtering,plot segments,None,list[int] or int or None,,Segments to emphasize or plot.
6,Selection/filtering,segment,None,int or None,,CV segment to analyze.
7,Selection/filtering,segments,None,list[int] or None,,One or more CV segments to analyze.
8,Units/normalization,C,None,float or None,,"Bulk concentration in mol/cm^3. If omitted, exact species metadata may be used when species is provided."
9,Units/normalization,D,None,float or None,,"Bulk diffusion coefficient in cm^2/s. If omitted, eCAT attempts branch-specific Sevcik Dapp estimates when area and concentration are available."


## Next Step

Notebook 08 covers the distinct surface-confined case. Notebook 09 then moves into normalization, catalytic analyses, and general fitting workflows.